# Belege (Extraction) Import via API

Imports data from `belege_*.xlsx` files into the `Extraction` model
using HTTP POST calls to the motm REST API.

**API base URL**: `http://localhost:8000/motm/api/`

Columns in the xlsx (row index 2 = header):
- `beleg_id` → `Extraction.identifier`
- `quelle` → source reference (informational)
- `interview_id` → linked `Interview.archive_id`
- `interview_datum` → (informational)
- `quelle_sprecher` → speaker `Person.identifier` → sets `Interview.interviewee`
- `betrifft_personen` → `Extraction.people_mentioned` (Person identifier)
- `timecode` → `Extraction.timecode`
- `themen` → `Extraction.concepts` (Concept label, comma-separated)
- `zitat` → `Extraction.quote`
- `markierung` → `Extraction.classification`
- `event_ids` → `Extraction.event` (comma-separated event IDs)
- `event_ids_confidence` → `Extraction.event_confidence`
- `notizen` → `Extraction.notes`

In [1]:
import os
from pathlib import Path

import requests
from openpyxl import load_workbook

In [ ]:
import getpass

BASE_URL = os.environ.get("MMT_API_URL", "http://localhost:8000/motm/api")
LOGIN_URL = os.environ.get("MMT_LOGIN_URL", "http://localhost:8000/admin/login/")
SESSION = requests.Session()

DATA_DIR = Path.cwd().parent / "data" / "schede mappatura"

# Authenticate via Django admin session
username = input("Admin username: ")
password = getpass.getpass("Admin password: ")
SESSION.get(LOGIN_URL)
SESSION.post(LOGIN_URL, data={
    "username": username,
    "password": password,
    "csrfmiddlewaretoken": SESSION.cookies["csrftoken"],
    "next": "/admin/",
})
SESSION.headers["X-CSRFToken"] = SESSION.cookies.get("csrftoken", "")
SESSION.headers["Referer"] = LOGIN_URL
print("Authenticated" if SESSION.cookies.get("sessionid") else "Login failed")

Authenticated


## API helpers

In [ ]:
def is_empty(value):
    return value in [None, "", "-", "–", "—"]


def clean(value):
    if value is None:
        return ""
    return str(value).strip()


def api_get(endpoint, params=None):
    """GET from the API, return JSON list (handles pagination)."""
    resp = SESSION.get(f"{BASE_URL}/{endpoint}/", params=params)
    resp.raise_for_status()
    data = resp.json()
    if isinstance(data, dict) and "results" in data:
        return data["results"]
    return data


def api_post(endpoint, payload):
    """POST to the API, return created object as dict."""
    resp = SESSION.post(f"{BASE_URL}/{endpoint}/", json=payload)
    resp.raise_for_status()
    return resp.json()


def api_patch(endpoint, obj_id, payload):
    """PATCH an existing object via the API, return updated object as dict."""
    resp = SESSION.patch(f"{BASE_URL}/{endpoint}/{obj_id}/", json=payload)
    resp.raise_for_status()
    return resp.json()

## Lookup helpers

In [4]:
def get_person_id(identifier):
    """Find a Person by identifier. Returns id or None."""
    if is_empty(identifier):
        return None
    existing = api_get("persons", params={"search": identifier})
    match = next((p for p in existing if p.get("identifier") == identifier), None)
    if match:
        return match["id"]
    return None


def get_interview_id(archive_id):
    """Find an Interview by archive_id. Returns id or None."""
    if is_empty(archive_id):
        return None
    existing = api_get("interviews", params={"search": archive_id})
    match = next((i for i in existing if i["archive_id"] == archive_id), None)
    if match:
        return match["id"]
    return None


def get_or_create_concept(label):
    """Get or create a Concept by label. Returns id."""
    label = label.strip()
    if is_empty(label):
        return None
    existing = api_get("concepts", params={"search": label})
    match = next((c for c in existing if c["label"] == label), None)
    if match:
        return match["id"]
    created = api_post("concepts", {"label": label})
    return created["id"]

## Read and import a belege xlsx

In [ ]:
def import_belege(xlsx_path, sheet_name="Belege"):
    """Import all rows from a belege xlsx file via the API."""
    wb = load_workbook(xlsx_path)
    sheet = wb[sheet_name]

    results = []

    for i, row in enumerate(sheet.iter_rows(values_only=True)):
        # Skip title (0), blank (1), header (2)
        if i < 3:
            continue

        if not row or all(cell is None for cell in row):
            continue

        cells = (list(row) + [None] * 13)[:13]
        beleg_id       = clean(cells[0])   # identifier
        # cells[1]: quelle (source reference, informational only)
        interview_id   = clean(cells[2])   # interview archive_id
        # cells[3]: interview_datum (informational only)
        quelle_sprecher = clean(cells[4])  # speaker person identifier → Interview.interviewee
        betrifft       = clean(cells[5])   # people mentioned (person identifiers)
        timecode       = clean(cells[6])   # timecode
        themen         = clean(cells[7])   # topics/concepts
        zitat          = clean(cells[8])   # quote
        markierung     = clean(cells[9])   # classification
        # cells[10]: event_ids (not yet linked — events must be imported first)
        event_confidence = clean(cells[11])
        notizen        = clean(cells[12])  # notes

        if is_empty(beleg_id):
            continue

        # Check if already imported
        existing = api_get("extractions", params={"search": beleg_id})
        if any(e["identifier"] == beleg_id for e in existing):
            print(f"  Skipping (exists): {beleg_id}")
            continue

        # Resolve person mentioned (first identifier from comma-separated list)
        person_id = None
        if betrifft:
            first_person_id = betrifft.split(",")[0].strip()
            person_id = get_person_id(first_person_id)

        # Resolve interview
        interview_db_id = get_interview_id(interview_id)

        # Link quelle_sprecher → Person → Interview.interviewee
        if not is_empty(quelle_sprecher) and interview_db_id:
            speaker_person_id = get_person_id(quelle_sprecher)
            if speaker_person_id:
                try:
                    api_patch("interviews", interview_db_id, {"interviewee": speaker_person_id})
                except Exception as e:
                    print(f"  ⚠ Could not set interviewee for interview {interview_id}: {e}")

        # Resolve first concept from comma-separated themen
        concept_id = None
        if themen:
            first_topic = themen.split(",")[0].strip()
            concept_id = get_or_create_concept(first_topic)

        payload = {
            "identifier": beleg_id,
            "timecode": timecode,
            "quote": zitat,
            "classification": markierung,
            "event_confidence": event_confidence,
            "notes": notizen,
        }
        if person_id:
            payload["people_mentioned"] = person_id
        if interview_db_id:
            payload["interview"] = interview_db_id
        if concept_id:
            payload["concepts"] = concept_id

        try:
            created = api_post("extractions", payload)
            results.append(created["id"])
            print(f"  ✅ {beleg_id}")
        except Exception as e:
            print(f"  ❌ {beleg_id}: {e}")

    print(f"\n✅ Imported {len(results)} extractions from {xlsx_path}")
    return results

## Import all belege files

In [6]:
def import_all_belege(directory=None):
    """Find and import all belege_*.xlsx files from the data directory."""
    if directory is None:
        directory = DATA_DIR
    directory = Path(directory)
    all_results = []
    for xlsx_path in sorted(directory.rglob("belege_*.xlsx")):
        print(f"\nProcessing: {xlsx_path.name}")
        try:
            ids = import_belege(xlsx_path)
            all_results.extend(ids)
        except Exception as e:
            print(f"\u274c ERROR: {e}")
    print(f"\n\u2705 Total imported: {len(all_results)} extractions")
    return all_results

## Run the import

Uncomment the appropriate line below.

In [8]:
# Single file:
import_belege(DATA_DIR / "stern_IS_S_00142" / "belege_josef_stern_v2.xlsx")

# All belege files:
# import_all_belege()

  ✅ bel_josef_001
  ✅ bel_josef_002
  ✅ bel_josef_003
  ✅ bel_josef_004
  ✅ bel_josef_005
  ✅ bel_josef_006
  ✅ bel_josef_007
  ✅ bel_josef_008
  ✅ bel_josef_009
  ✅ bel_josef_010
  ✅ bel_josef_011
  ✅ bel_josef_012
  ✅ bel_josef_013
  ✅ bel_josef_014
  ✅ bel_josef_015
  ✅ bel_josef_016
  ✅ bel_josef_017
  ✅ bel_josef_018
  ✅ bel_josef_019
  ✅ bel_josef_020
  ✅ bel_josef_021
  ✅ bel_josef_022
  ✅ bel_josef_023
  ✅ bel_josef_024
  ✅ bel_josef_025
  ✅ bel_josef_026
  ✅ bel_josef_027
  ✅ bel_josef_028
  ✅ bel_josef_029
  ✅ bel_josef_030
  ✅ bel_josef_031
  ✅ bel_josef_032
  ✅ bel_josef_033
  ✅ bel_josef_034
  ✅ bel_josef_035
  ✅ bel_josef_036
  ✅ bel_josef_037
  ✅ bel_josef_038
  ✅ bel_josef_039
  ✅ bel_josef_040
  ✅ bel_josef_041
  ✅ bel_josef_042
  ✅ bel_josef_043
  ✅ bel_josef_044
  ✅ bel_josef_045
  ✅ bel_josef_046
  ✅ bel_josef_047
  ✅ bel_josef_048
  ✅ bel_josef_049
  ✅ bel_josef_050
  ✅ bel_josef_051
  ✅ bel_josef_052
  ✅ bel_josef_053
  ✅ bel_josef_054
  ✅ bel_josef_055
  ✅ bel_jo

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130]